In [1]:
from pathlib import Path

from doc_processor import (
	Annotation,
	ParserConfig,
	ParagraphTextEdit,
	RunTextEdit,
	apply_edits_to_doc_ir,
	apply_edits_to_file,
	render_annotated_html,
	run_parser,
)
from doc_processor.types import RelevanceMode

from os import listdir, environ

from langfuse import get_client
from langfuse._client.resource_manager import LangfuseResourceManager

get_client().shutdown()
LangfuseResourceManager.reset()

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


In [2]:
environ["LANGFUSE_TIMEOUT"] = "30"

print(environ.get("LANGFUSE_TIMEOUT"))
print(environ.get("LANGFUSE_FLUSH_INTERVAL"))
print(environ.get("LANGFUSE_FLUSH_AT"))
print(environ.get("LANGFUSE_HOST"))

30
None
None
None


In [3]:
doc_dir = Path("doc_samples") / "표준계약서모음(hwp-hwpx)"
doc_out = Path("results")

# Change this index or point `target` directly at a file while testing.
target = doc_dir / sorted(listdir(doc_dir))[2]

config = ParserConfig(
    relevance_mode=RelevanceMode.KEYWORD_ONLY,
    boundary_review_enabled=True,
    label_review_enabled=True,
    langfuse_enabled=True,
    max_concurrent_workers=10
)

target

PosixPath('doc_samples/표준계약서모음(hwp-hwpx)/03. 출판권 설정 계약서.hwp')

In [4]:
state = run_parser(Path(target), config=config)

# state = run_parser(
# 	Path(target),
# 	config=ParserConfig(
# 		boundary_review_enabled=False,
# 		label_review_enabled=False,
# 	),
# )

2026-04-14 16:26:41,518 | INFO | structure analysis run start source=doc_samples/표준계약서모음(hwp-hwpx)/03. 출판권 설정 계약서.hwp
2026-04-14 16:26:41,556 | INFO | [structure_analysis.load_document] start


2026-04-14 16:26:42,100 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-04-14 16:26:42,115 | INFO | [structure_analysis.screen_relevance] start
2026-04-14 16:26:42,117 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-04-14 16:26:42,119 | INFO | [structure_analysis.regex_analysis] start
2026-04-14 16:26:42,122 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-04-14 16:26:42,128 | INFO | [structure_analysis.llm_analysis] start
2026-04-14 16:26:42,128 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (31 suspects)
2026-04-14 16:26:42,128 | INFO | [structure_analysis.llm_analysis] done goto=boundary_llm_batch
2026-04-14 16:26:42,134 | INFO | [structure_analysis.llm_analysis.boundary_batch] start
2026-04-14 16:26:48,018 | INFO | [structure_analysis.llm_analysis.boundary_batch] complete (31 reviews)
2026-04-14 16:26:48,019 | INFO | [structure_analysis.llm_analysis.boundary_batch] done go

In [5]:
doc = state.working_doc

# Inspect ids first
for paragraph in doc.paragraphs[:5]:
	print(paragraph.unit_id, paragraph.text)
	for run in paragraph.runs:
		print("  ", run.unit_id, repr(run.text))

annotations = [
	Annotation(
		target_unit_id="s1.p3",   # paragraph-level highlight
		start=0,
		end=8,
		label="Risk",
		color="#FFDD88",
		note="Check this clause",
	),
	Annotation(
		target_unit_id="s1.p3.r1",  # run-level highlight
		start=0,
		end=4,
		label="Term",
		color="#99EEFF",
	),
]

s1.p1 [별표1]
   s1.p1.r1 '[별표1]'
s1.p2 
   s1.p2.r1 ''
s1.p3 출판권 설정 계약서(제2조 제1호 관련)
   s1.p3.r1 '출판권 설정 계약서(제2조 제1호 관련)'
s1.p4 저작권자1) _____(이하 ‘저작권자’라고 한다)와(과) 출판권자 _____(이하 ‘출판권자’라고 한다)는(은) 아래 대상 저작물에 대하여 아래와 같이 출판권설정계약을 체결한다.

대상 저작물의 표시
제호(가제) :

1) 저작권자는 저작재산권을 보유한 ‘저작자’와 저작재산권을 양도받아 권리를 행사하는 ‘저작재산권자’를 포함한다.
s1.p5 
   s1.p5.r1 ''


In [6]:
html = render_annotated_html(doc, annotations, title="Review")

In [7]:
from IPython.display import HTML, display
display(HTML(html))

저작권자1) _____(이하 ‘저작권자’라고 한다)와(과) 출판권자 _____(이하 ‘출판권자’라고 한다)는(은) 아래 대상 저작물에 대하여 아래와 같이 출판권설정계약을 체결한다. 대상 저작물의 표시제호(가제) : 1) 저작권자는 저작재산권을 보유한 ‘저작자’와 저작재산권을 양도받아 권리를 행사하는 ‘저작재산권자’를 포함한다.
비독점적 성격의 권리 설정 시에는 본 계약서가 아닌 타 계약서(출판 이용허락계약 등)를 활용하여야 한다.
"거래안전을 위해 ‘지체없이’가 원칙이나, 합의에 따라서 일 수를 정할 수 있다."
"이 계약기간 중에 위 저작물에 대하여 국내외 제3자가 2차적저작물로서의 이용을 요청할 때, ‘출판권자’에게 먼저 요청이 오는 경우 ‘출판권자’는 제3자의 저작물 이용허락 요청 사실을 ‘저작권자’에게 알려주어야 한다."
"민법 등 관계 법령이 있는 경우에는 법정 해지·해제, 합의 해지·해제할 수 있다."
